# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO N.° 11 — Analítica Prescriptiva: Optimización

**Semana 11 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía || **Alumno:** Rodrigo Gogin Cisterna  
**Dataset real:** Online Retail II — UCI Machine Learning Repository  
**Herramienta:** PuLP

### Pregunta de negocio
**¿Cómo debe distribuirse una capacidad limitada de inventario entre productos para maximizar el valor potencial de venta sin descuidar niveles mínimos de servicio?**

> El objetivo no es solo obtener `Optimal`. Debe demostrarse cuánto mejora la decisión frente a una regla intuitiva.

## Identificación
- Sección: C28
- Integrantes: Rodrigo Alejandro Gogin Cisterna
- Repositorio: https://github.com/RGogin28/analitica-empresarial-integrada.
- Uso de IA: No se uso ia ni para el desarrollo, ni para redactar interpretaciones ni respuestas a las preguntas reto planteadas


In [1]:
# Instala en Colab las librerías que no vienen por defecto; la versión de PuLP queda fija para repetir resultados
%pip install -q "ucimlrepo>=0.0.7" "pulp==3.3.0" plotly openpyxl

In [2]:
# Librerías: pandas y numpy para los datos, PuLP para optimizar y plotly para graficar
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import pulp as pl
from ucimlrepo import fetch_ucirepo
# Se registran las versiones para que otra persona pueda repetir el resultado
print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· PuLP:", pl.__version__)
# Permite ver hasta 50 columnas al imprimir tablas
pd.set_option("display.max_columns", 50)

Python: 3.13.18 · pandas: 2.2.3 · PuLP: 3.3.0


# 1. Descarga de datos reales y trazabilidad

In [3]:
# API UCI; si falla, usar el ZIP oficial. La carga manual queda como último recurso.
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
from pathlib import Path

try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
    fuente = "API ucimlrepo, UCI ID 502"
except Exception as exc:
    print("API no disponible:", type(exc).__name__)
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as response:
            with ZipFile(BytesIO(response.read())) as z:
                name = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
                df = pd.concat(pd.read_excel(BytesIO(z.read(name)), sheet_name=None).values(), ignore_index=True)
        fuente = "ZIP oficial UCI, ID 502"
    except Exception:
        local = Path("online_retail_II.xlsx")
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a Colab.")
        df = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)
        fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente, "· ejecución UTC:", pd.Timestamp.now(tz="UTC").isoformat())
print("Dimensión:", df.shape, "· columnas:", df.columns.tolist())
display(df.head())

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502 · ejecución UTC: 2026-10-29T01:44:24.611544+00:00
Dimensión: (1067371, 8) · columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Normalización de nombres

In [4]:
# El archivo trae nombres de columna distintos según la fuente (Invoice o InvoiceNo, Price o UnitPrice); esta función busca cuál se usa
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")
        if key in normalized:
            return normalized[key]
    return None

# Se traduce cada columna a un nombre único que usa el resto del notebook
mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "Country": find_col(df.columns, ["Country"]),
}

# Si falta una columna indispensable, se detiene aquí en lugar de fallar más adelante
required = ["InvoiceNo","StockCode","Quantity","InvoiceDate","UnitPrice","Country"]
missing = [k for k in required if mapping[k] is None]
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")

# Se renombran las columnas y se muestra la correspondencia encontrada
df = df.rename(columns={v:k for k,v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'Country': 'Country'}


# 2. Control de calidad

In [5]:
# Se convierten fecha, cantidad y precio a su tipo correcto; lo que no se pueda convertir queda vacío
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
# Se quitan los registros sin código de producto y se limpian espacios
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
# Las facturas que empiezan con C son cancelaciones: restan demanda y no deben contarse como ventas
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
# Regla de limpieza: solo ventas válidas (cantidad y precio positivos, con fecha, producto y país)
clean = df.loc[(~cancel) & df["InvoiceDate"].notna() &
               df["StockCode"].ne("") & df["Country"].notna() &
               (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
# Se conserva el Reino Unido, el mercado principal del archivo
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
# Se registra cuántas cancelaciones había y cuántas filas válidas quedaron
print("Cancelaciones:", int(cancel.sum()), "· filas válidas UK:", len(uk))
print("Periodo:", uk["InvoiceDate"].min(), "→", uk["InvoiceDate"].max())

Cancelaciones: 19494 · filas válidas UK: 958501
Periodo: 2009-12-01 07:45:00 → 2011-12-09 12:49:00


### Pregunta 1
¿Por qué es incorrecto usar directamente la suma de Quantity sin revisar cancelaciones y precios?

**Respuesta:**

Porque `Quantity` mezcla registros que no son demanda comercial:

- **Cancelaciones y devoluciones.** Las facturas que empiezan con `C` (19,494 filas) traen cantidades negativas. Si se suman sin identificarlas, el neto oculta lo que realmente se pidió; si se tratan como ventas, se invierte el signo del efecto. En ambos casos la demanda queda mal medida.
- **Precios 0 o negativos.** Corresponden a ajustes, muestras, mercadería dañada o correcciones manuales: son unidades sin valor de venta. Inflan la demanda y, al multiplicar por precio, distorsionan el valor potencial que el modelo quiere maximizar.
- **Registros incompletos.** Sin fecha, producto o país no se puede ubicar la venta en la ventana de 12 semanas ni en el mercado analizado.

Por eso se aplicó la regla de limpieza (sin cancelaciones, `Quantity > 0`, `UnitPrice > 0`, con fecha, producto y país) y se conservó el Reino Unido: de 1,067,371 filas quedaron 958,501 válidas. Como la capacidad se define como porcentaje de la demanda, un error en la suma de `Quantity` se propaga a la capacidad, a los mínimos de servicio y a todo el modelo.

# 3. Construcción de parámetros reales

In [6]:
# Últimas 12 semanas COMPLETAS: lunes a domingo.
# El archivo termina el 9-dic-2011 (viernes); se excluye esa semana parcial.
last_obs = uk["InvoiceDate"].max().normalize()
last_complete_sunday = last_obs - pd.Timedelta(days=last_obs.weekday() + 1)
first_monday = last_complete_sunday - pd.Timedelta(weeks=12) + pd.Timedelta(days=1)
recent = uk.loc[
    (uk["InvoiceDate"] >= first_monday) &
    (uk["InvoiceDate"] < last_complete_sunday + pd.Timedelta(days=1))
].copy()
recent["week"] = recent["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
assert recent["week"].nunique() == 12, "Revisar cobertura de las 12 semanas."

weekly = recent.groupby(["StockCode", "week"], as_index=False).agg(
    units=("Quantity", "sum"), price=("UnitPrice", "median")
)
sku_params = weekly.groupby("StockCode").agg(
    weeks_present=("week", "nunique"),
    demand_total=("units", "sum")
).reset_index()
sku_params["weekly_demand"] = sku_params["demand_total"] / 12
sku_params["price"] = sku_params["StockCode"].map(
    recent.groupby("StockCode")["UnitPrice"].median()
)
eligible = sku_params.loc[sku_params["weeks_present"] >= 8].copy()
if len(eligible) < 10:
    eligible = sku_params.loc[sku_params["weeks_present"] >= 6].copy()
if len(eligible) < 10:
    raise ValueError("Menos de 10 SKU elegibles; revisar datos y regla de selección.")
top = eligible.sort_values(["demand_total", "StockCode"], ascending=[False, True]).head(10).copy()
top["demanda"] = np.maximum(np.ceil(top["weekly_demand"]), 1).astype(int)
parametros = top.set_index("StockCode")[["demanda", "price", "weeks_present", "demand_total"]].copy()
print("Ventana:", first_monday.date(), "a", last_complete_sunday.date())
display(parametros)
print("Demanda esperada total:", int(parametros["demanda"].sum()))

Ventana: 2011-09-12 a 2011-12-04


,demanda,price,weeks_present,demand_total
StockCode,,,,
22197,1809,0.85,12,21703
84077,1223,0.29,12,14676
85099B,1175,2.08,12,14098
22086,1080,2.95,12,12951
23084,959,2.08,12,11505
84879,860,1.69,12,10310
22952,765,0.55,12,9177
22577,746,0.29,12,8948
22578,693,0.29,12,8305


Demanda esperada total: 9982


### Pregunta 2
¿Qué significa una semana sin ventas observadas para un SKU y por qué no prueba demanda cero?

**Respuesta:**

Significa que en esa semana **no quedó registrada ninguna transacción válida** de ese SKU en el archivo. Solo observamos ventas, no la demanda real.

No prueba demanda cero porque la ausencia de ventas pudo deberse a:

- **Quiebre de stock:** si no había producto, no hubo venta aunque existieran clientes dispuestos a comprar (demanda censurada).
- **Compras mayoristas irregulares:** pocos pedidos grandes y espaciados generan semanas vacías sin que la demanda sea nula.
- **Estacionalidad o precio:** el producto pudo no estar ofrecido o tener un precio poco atractivo esa semana.
- **Limpieza de datos:** registros descartados (cancelaciones, precio 0) que no cuentan como venta.

Por eso el notebook exige ventas en al menos 8 de las 12 semanas y divide el total entre 12 semanas completas: las semanas vacías entran como cero, un supuesto conservador y no una medición de demanda cero. Los 10 SKU elegidos tienen `weeks_present = 12`, así que aquí el efecto es mínimo, pero debe tenerse presente al extrapolar.

# 4. Supuestos explícitos del escenario

In [7]:
# Supuestos académicos del caso (no vienen de UCI): capacidad, servicio mínimo y concentración máxima
CAPACITY_RATIO = 0.70
MIN_SERVICE_RATIO = 0.20
MAX_CONCENTRATION_RATIO = 0.35
# Se derivan los límites en unidades enteras a partir de la demanda estimada
demanda_total = int(parametros["demanda"].sum())
capacidad_total = int(np.floor(CAPACITY_RATIO * demanda_total))
limite_concentracion = int(np.floor(MAX_CONCENTRATION_RATIO * capacidad_total))
# Cada SKU debe recibir como mínimo el 20 % de su demanda (y al menos una unidad)
parametros["min_servicio"] = np.maximum(
    np.ceil(MIN_SERVICE_RATIO * parametros["demanda"]).astype(int), 1
)
# Se comprueba que el problema tiene solución: los mínimos caben en la capacidad y en el tope por SKU
assert parametros["min_servicio"].sum() <= capacidad_total
assert (parametros["min_servicio"] <= limite_concentracion).all()
# Se muestran la demanda y la capacidad resultantes, y la tabla de parámetros
print("Demanda:", demanda_total, "· capacidad:", capacidad_total)
display(parametros)

Demanda: 9982 · capacidad: 6987


,demanda,price,weeks_present,demand_total,min_servicio
StockCode,,,,,
22197,1809,0.85,12,21703,362
84077,1223,0.29,12,14676,245
85099B,1175,2.08,12,14098,235
22086,1080,2.95,12,12951,216
23084,959,2.08,12,11505,192
84879,860,1.69,12,10310,172
22952,765,0.55,12,9177,153
22577,746,0.29,12,8948,150
22578,693,0.29,12,8305,139


> **Importante:** 70 % de capacidad, 20 % de servicio mínimo y 35 % de concentración son supuestos académicos del caso. No provienen de UCI.

### Pregunta 3
Separe parámetros derivados de UCI y supuestos de operaciones.

**Respuesta:**

**Parámetros derivados de UCI (Online Retail II):**

- Mercado: Reino Unido, ventas válidas (sin cancelaciones, cantidad y precio positivos).
- Ventana: 12 semanas completas, del 2011-09-12 al 2011-12-04.
- Selección: los 10 SKU con mayor `demand_total` entre los que tienen ventas en al menos 8 semanas.
- Demanda semanal por SKU = total de 12 semanas ÷ 12, redondeado hacia arriba. Demanda total = 9,982 unidades.
- Precio por SKU = mediana del precio unitario en la ventana (por ejemplo £0.29 en 84077 y £2.95 en 22086).

**Supuestos de operaciones (académicos, no provienen de UCI):**

- Capacidad = 70 % de la demanda → 6,987 unidades (escenarios alternos: 60 % y 85 %).
- Servicio mínimo = 20 % de la demanda de cada SKU → 1,999 unidades en total.
- Concentración máxima = 35 % de la capacidad → 2,445 unidades por SKU.
- Objetivo = maximizar precio × unidades (valor potencial de venta, sin costos ni márgenes).
- Cada SKU no puede recibir más que su demanda estimada, y las unidades son enteras.

# 5. Línea base intuitiva: asignación proporcional a la demanda

In [8]:
# Línea base intuitiva: reparte la capacidad en proporción a la demanda, respetando mínimos, demanda y tope por SKU
def proportional_allocation(params, capacity, min_col="min_servicio", max_conc=None):
    """Regla proporcional con mínimos, demanda y concentración: baseline factible."""
    # Tope de cada SKU: su demanda y, si existe, el límite de concentración
    upper = params["demanda"].astype(int).copy()
    if max_conc is not None:
        upper = upper.clip(upper=int(max_conc))
    # Piso de cada SKU: el servicio mínimo
    lower = params[min_col].astype(int).copy()
    # Si las reglas no permiten asignar toda la capacidad, se avisa en lugar de comparar algo injusto
    if (lower > upper).any() or lower.sum() > capacity or upper.sum() < capacity:
        raise ValueError("La política no permite asignar toda la capacidad.")
    # Se parte de los mínimos y se reparte el resto en proporción a la demanda
    alloc = lower.copy()
    remaining = int(capacity - alloc.sum())
    weights = params["demanda"].astype(float)
    # Se repite hasta repartir todas las unidades, respetando el espacio que le queda a cada SKU
    while remaining:
        room = upper - alloc
        active = room > 0
        if not active.any():
            break
        quota = remaining * weights[active] / weights[active].sum()
        add = np.minimum(np.floor(quota).astype(int), room[active])
        if add.sum() == 0:
            order = (quota - np.floor(quota)).sort_values(ascending=False).index
            for sku in order:
                if remaining == 0: break
                if room.loc[sku] > 0:
                    alloc.loc[sku] += 1
                    remaining -= 1
        else:
            alloc.loc[active] += add
            remaining -= int(add.sum())
    # Verificación final: se asignó exactamente la capacidad
    assert alloc.sum() == capacity
    return alloc

# Se aplica la regla y se calcula su valor potencial: es la referencia que PuLP debe superar
baseline = parametros.copy()
baseline["asignacion_intuitiva"] = proportional_allocation(
    parametros, capacidad_total, max_conc=limite_concentracion
)
baseline["valor_intuitivo"] = baseline["asignacion_intuitiva"] * baseline["price"]
display(baseline)
print("Unidades intuitivas:", int(baseline["asignacion_intuitiva"].sum()))
print("Valor potencial intuitivo (£):", round(baseline["valor_intuitivo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_intuitiva,valor_intuitivo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1266,1076.10
84077,1223,0.29,12,14676,245,857,248.53
85099B,1175,2.08,12,14098,235,823,1711.84
22086,1080,2.95,12,12951,216,756,2230.20
23084,959,2.08,12,11505,192,672,1397.76
84879,860,1.69,12,10310,172,601,1015.69
22952,765,0.55,12,9177,153,535,294.25
22577,746,0.29,12,8948,150,522,151.38
22578,693,0.29,12,8305,139,485,140.65


Unidades intuitivas: 6987
Valor potencial intuitivo (£): 8853.9


### Pregunta 4
¿Por qué la comparación con PuLP sería engañosa si la regla intuitiva excediera un límite?

**Respuesta:**

Porque dejaría de ser una comparación entre dos decisiones **posibles bajo las mismas reglas**. Si la regla intuitiva excediera la capacidad, la demanda, el servicio mínimo o el tope de concentración, su valor sería el de una solución que la empresa no puede ejecutar. Habría dos efectos engañosos:

- Si la regla viola un límite y aun así obtiene menos valor, la mejora de PuLP se exagera, porque PuLP cumple condiciones más exigentes.
- Si la regla viola un límite y obtiene más valor, el modelo parecería peor sin serlo, porque el valor extra viene de incumplir una restricción.

Por eso `proportional_allocation` parte de los mínimos, reparte el resto en proporción a la demanda, respeta el tope por SKU (demanda y concentración), verifica que asigna exactamente la capacidad y lanza un error si las reglas no permiten una asignación factible. Así la mejora de £2,624.13 (29.64 %) mide solo la calidad de la decisión.

# 6. Formulación con PuLP

In [9]:
# Modelo: maximizar el valor potencial de venta (precio × unidades asignadas)
model = pl.LpProblem("Asignacion_Inventario", pl.LpMaximize)
# Variable de decisión: unidades enteras (mayores o iguales a 0) que se asignan a cada SKU
x = {sku: pl.LpVariable(f"x_{i}", lowBound=0, cat=pl.LpInteger)
     for i, sku in enumerate(parametros.index)}
# Función objetivo
model += pl.lpSum(parametros.loc[sku, "price"] * x[sku] for sku in parametros.index)
# Restricción de capacidad: no se puede asignar más de lo que hay
model += pl.lpSum(x.values()) <= capacidad_total
# Por cada SKU: no más que su demanda, no menos que su servicio mínimo y no más que el tope de concentración
for sku in parametros.index:
    model += x[sku] <= int(parametros.loc[sku, "demanda"])
    model += x[sku] >= int(parametros.loc[sku, "min_servicio"])
    model += x[sku] <= limite_concentracion
# Con 10 SKU se esperan 10 variables y 31 restricciones (1 de capacidad y 3 por SKU)
print("Variables:", len(x), "· restricciones:", len(model.constraints))

Variables: 10 · restricciones: 31


### Pregunta 5
Explique por qué maximizar precio × unidades sin restricciones no responde la decisión de negocio.

**Respuesta:**

Porque sin restricciones el problema no representa ninguna decisión de negocio:

- **No tiene límite:** con unidades enteras y `x ≥ 0`, maximizar precio × unidades crece sin tope. PuLP devolvería `Unbounded`, ya que siempre conviene asignar una unidad más.
- **Ignora la capacidad:** el enunciado habla de una capacidad *limitada* de inventario (6,987 unidades); sin esa restricción no hay nada que distribuir.
- **Ignora la demanda:** no tiene sentido asignar más unidades de las que se pueden vender (por ejemplo, más de 1,080 unidades de 22086).
- **Ignora el servicio mínimo:** la pregunta pide no descuidar niveles mínimos de servicio; sin ellos el modelo dejaría en cero a los SKU baratos.
- **Ignora la concentración:** podría poner todo el inventario en el SKU más caro (22086, £2.95).

Las restricciones son las que convierten una fórmula en una política: capacidad, demanda, servicio mínimo y concentración son las reglas del negocio, y la función objetivo solo dice qué se quiere maximizar dentro de ellas.

# 7. Resolver el problema

In [10]:
# Se elige el solucionador CBC, que viene con PuLP, y se resuelve el modelo
solver = pl.PULP_CBC_CMD(msg=False)
status = model.solve(solver)

# El estado dice si hay solución (Optimal, Infeasible, Unbounded o Not Solved); se lee antes que los números
print("Estado:", pl.LpStatus[status])
print("Valor objetivo (£):", round(pl.value(model.objective), 2))

Estado: Optimal
Valor objetivo (£): 11478.03


In [11]:
# No se sigue si el solucionador no encontró un óptimo
if pl.LpStatus[status] != "Optimal":
    raise RuntimeError(
        "El modelo no alcanzó estado Optimal. Revise capacidad, mínimos y concentración."
    )

# Se guarda la asignación óptima por SKU y su valor potencial
opt = parametros.copy()
opt["asignacion_optima"] = [int(round(x[sku].value())) for sku in opt.index]
opt["valor_optimo"] = opt["asignacion_optima"] * opt["price"]

# Se muestran la tabla y los totales: las unidades asignadas deben igualar la capacidad
display(opt)
print("Unidades asignadas:", int(opt["asignacion_optima"].sum()))
print("Valor potencial óptimo (£):", round(opt["valor_optimo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_optima,valor_optimo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1554,1320.90
84077,1223,0.29,12,14676,245,245,71.05
85099B,1175,2.08,12,14098,235,1175,2444.00
22086,1080,2.95,12,12951,216,1080,3186.00
23084,959,2.08,12,11505,192,959,1994.72
84879,860,1.69,12,10310,172,860,1453.40
22952,765,0.55,12,9177,153,153,84.15
22577,746,0.29,12,8948,150,150,43.50
22578,693,0.29,12,8305,139,139,40.31


Unidades asignadas: 6987
Valor potencial óptimo (£): 11478.03


### Pregunta 6
¿Qué garantiza el estado Optimal y qué no garantiza respecto de la demanda y los costos?

**Respuesta:**

**Lo que garantiza `Optimal`:** que, dado el modelo (función objetivo, restricciones y parámetros), el solucionador CBC encontró una solución factible con el mayor valor posible dentro de todas las que cumplen las restricciones. En este caso: £11,478.03 con las 6,987 unidades asignadas.

**Lo que no garantiza:**

- **Demanda:** que la demanda estimada sea correcta. Es el promedio de ventas observadas en 12 semanas (temporada alta, septiembre a diciembre), no demanda real ni futura.
- **Costos y utilidad:** el modelo maximiza precio × unidades, no margen. No incluye costo de compra, almacenamiento, obsolescencia ni costo de faltante, así que valor potencial ≠ utilidad.
- **Precios constantes:** supone que el precio mediano se mantiene y que vender más unidades no altera la demanda.
- **Parámetros del caso:** que 70 %, 20 % y 35 % sean los valores correctos; son supuestos académicos.
- **Resultado real:** que las ventas efectivamente ocurran.

`Optimal` significa óptimo **para el modelo**, no óptimo para la realidad.

# 8. Comparación: intuitivo vs óptimo

In [12]:
# Se juntan en una tabla la asignación intuitiva y la óptima, por SKU
comparison = parametros[["demanda","price"]].copy()
comparison["intuitivo"] = baseline["asignacion_intuitiva"]
comparison["optimo"] = opt["asignacion_optima"]
# Diferencia de unidades: positivo significa que el modelo asigna más que la regla intuitiva
comparison["dif_unidades"] = comparison["optimo"] - comparison["intuitivo"]
comparison["valor_intuitivo"] = comparison["intuitivo"] * comparison["price"]
comparison["valor_optimo"] = comparison["optimo"] * comparison["price"]

# Se ordena de mayor a menor cambio para ver quién gana y quién pierde unidades
display(comparison.sort_values("dif_unidades", ascending=False))

,demanda,price,intuitivo,optimo,dif_unidades,valor_intuitivo,valor_optimo
StockCode,,,,,,,
85099B,1175,2.08,823,1175,352,1711.84,2444.00
22086,1080,2.95,756,1080,324,2230.20,3186.00
22197,1809,0.85,1266,1554,288,1076.10,1320.90
23084,959,2.08,672,959,287,1397.76,1994.72
84879,860,1.69,601,860,259,1015.69,1453.40
21915,672,1.25,470,672,202,587.50,840.00
22578,693,0.29,485,139,-346,140.65,40.31
22577,746,0.29,522,150,-372,151.38,43.50
22952,765,0.55,535,153,-382,294.25,84.15


In [13]:
# Valor total de cada solución
v_int = float(comparison["valor_intuitivo"].sum())
v_opt = float(comparison["valor_optimo"].sum())
# Mejora absoluta (en £) y relativa (frente a la regla intuitiva)
improvement = v_opt - v_int
improvement_pct = improvement / v_int if v_int else np.nan

# Se informa siempre en £ y en porcentaje: es valor potencial de venta, no utilidad
print("Valor intuitivo (£):", round(v_int, 2))
print("Valor óptimo (£):", round(v_opt, 2))
print("Mejora absoluta (£):", round(improvement, 2))
print("Mejora porcentual:", f"{improvement_pct:.2%}")

Valor intuitivo (£): 8853.9
Valor óptimo (£): 11478.03
Mejora absoluta (£): 2624.13
Mejora porcentual: 29.64%


In [14]:
# Se reordena la tabla en formato largo para graficar los dos métodos lado a lado
long = (
    comparison[["intuitivo","optimo"]]
    .reset_index()
    .melt(id_vars="StockCode", var_name="metodo", value_name="unidades")
)

# Barras agrupadas: unidades asignadas por SKU, regla intuitiva frente a óptimo
fig = px.bar(
    long,
    x="StockCode",
    y="unidades",
    color="metodo",
    barmode="group",
    title="Asignación intuitiva vs óptima"
)
fig.show()

### Pregunta 7
¿Qué SKU ganan y pierden unidades, y cuál es la lógica económica de ese cambio?

**Respuesta:**

**Ganan unidades (6 SKU, 1,712 unidades en total):** 85099B (+352), 22086 (+324), 22197 (+288), 23084 (+287), 84879 (+259) y 21915 (+202). Sus precios son £2.08, £2.95, £0.85, £2.08, £1.69 y £1.25.

**Pierden unidades (4 SKU, −1,712 unidades):** 84077 (−612), 22952 (−382), 22577 (−372) y 22578 (−346). Sus precios son £0.29, £0.55, £0.29 y £0.29, los más bajos de la lista.

**Lógica económica:** cada unidad de capacidad vale lo que cuesta el SKU que la ocupa. La regla proporcional reparte según demanda y por eso gasta capacidad escasa en productos baratos de alta rotación. El modelo mueve 1,712 unidades desde los SKU de £0.29 a £0.55 hacia los de mayor precio, hasta que estos alcanzan su demanda (85099B, 22086, 23084, 84879 y 21915 quedan exactamente en su demanda). Las unidades restantes van a 22197 (£0.85), que llega a 1,554 de 1,809. Los SKU baratos bajan hasta su servicio mínimo (20 % de la demanda). Resultado: +£2,624.13 (+29.64 %) con la misma capacidad.

# 9. Verificación de restricciones

In [15]:
# Holgura = límite − uso: cuánto sobra en cada regla; 0 significa restricción activa
checks = pd.DataFrame(index=parametros.index)
checks["x"] = opt["asignacion_optima"]
checks["demanda"] = parametros["demanda"]
checks["min_servicio"] = parametros["min_servicio"]
checks["holgura_demanda"] = checks["demanda"] - checks["x"]
checks["holgura_minimo"] = checks["x"] - checks["min_servicio"]
checks["holgura_concentracion"] = limite_concentracion - checks["x"]
display(checks)
# Verificaciones: no pasar la capacidad y ninguna holgura negativa (sería una regla violada)
assert checks["x"].sum() <= capacidad_total
assert (checks[["holgura_demanda","holgura_minimo","holgura_concentracion"]] >= 0).all().all()
# Resumen: capacidad usada y si la concentración es vinculante (holgura 0 en algún SKU)
print("Capacidad usada:", int(checks["x"].sum()), "/", capacidad_total)
print("Límite de concentración vinculante:", bool((checks["holgura_concentracion"] == 0).any()))

,x,demanda,min_servicio,holgura_demanda,holgura_minimo,holgura_concentracion
StockCode,,,,,,
22197,1554,1809,362,255,1192,891
84077,245,1223,245,978,0,2200
85099B,1175,1175,235,0,940,1270
22086,1080,1080,216,0,864,1365
23084,959,959,192,0,767,1486
84879,860,860,172,0,688,1585
22952,153,765,153,612,0,2292
22577,150,746,150,596,0,2295
22578,139,693,139,554,0,2306


Capacidad usada: 6987 / 6987
Límite de concentración vinculante: False


### Pregunta 8
¿La concentración del 35 % es vinculante en el escenario base? Sustente con las holguras.

**Respuesta:**

**No es vinculante.** El límite del 35 % equivale a `floor(0.35 × 6,987) = 2,445` unidades por SKU. La asignación más grande es 22197 con 1,554 unidades (22.2 % de la capacidad), de modo que la menor holgura de concentración es **891 unidades** (22197) y las demás van de 1,270 a 2,306. Ninguna holgura es 0, y la verificación imprime `Límite de concentración vinculante: False`.

Las restricciones que sí están activas (holgura 0) son:

- **Capacidad:** se usan 6,987 de 6,987 unidades.
- **Demanda:** 85099B, 22086, 23084, 84879 y 21915 reciben exactamente su demanda.
- **Servicio mínimo:** 84077, 22952, 22577 y 22578 reciben exactamente su mínimo.

Por tanto, en el escenario base la concentración no limita el valor: quitarla o relajarla no mejoraría el resultado.

# 10. Sensibilidad a la capacidad

In [16]:
# Resuelve el mismo modelo con otros supuestos: devuelve el estado, el valor óptimo, el valor de la línea base y la asignación
def solve_scenario(params, capacity_ratio=0.70, min_service_ratio=0.20,
                   max_conc_ratio=0.35):
    # Se recalculan la capacidad y el tope de concentración del escenario
    total = int(params["demanda"].sum())
    capacity = int(np.floor(capacity_ratio * total))
    max_conc = int(np.floor(max_conc_ratio * capacity))
    p = params.copy()
    # Servicio mínimo del escenario (al menos una unidad por SKU)
    p["min_scn"] = np.maximum(np.ceil(min_service_ratio * p["demanda"]).astype(int), 1)
    # Si los mínimos no caben, el escenario es infactible y no se resuelve
    if p["min_scn"].sum() > capacity or (p["min_scn"] > max_conc).any():
        return {"status":"Infeasible", "capacity":capacity, "optimal_value":np.nan,
                "baseline_value":np.nan, "uplift_pct":np.nan, "allocation":None}
    # Mismo modelo entero del laboratorio, con los límites del escenario
    m = pl.LpProblem("Escenario", pl.LpMaximize)
    z = {sku: pl.LpVariable(f"z_{i}", lowBound=0, cat=pl.LpInteger)
         for i,sku in enumerate(p.index)}
    m += pl.lpSum(p.loc[sku,"price"]*z[sku] for sku in p.index)
    m += pl.lpSum(z.values()) <= capacity
    for sku in p.index:
        m += z[sku] <= int(p.loc[sku,"demanda"])
        m += z[sku] >= int(p.loc[sku,"min_scn"])
        m += z[sku] <= max_conc
    # Se resuelve y se lee el estado antes de usar los valores
    status = pl.LpStatus[m.solve(pl.PULP_CBC_CMD(msg=False))]
    if status != "Optimal":
        return {"status":status,"capacity":capacity,"optimal_value":np.nan,
                "baseline_value":np.nan,"uplift_pct":np.nan,"allocation":None}
    # Asignación óptima y línea base proporcional bajo las mismas reglas, para comparar con justicia
    allocation = pd.Series({sku:int(round(z[sku].value())) for sku in p.index})
    baseline_scn = proportional_allocation(p, capacity, min_col="min_scn", max_conc=max_conc)
    optimal_value = float((allocation*p["price"]).sum())
    baseline_value = float((baseline_scn*p["price"]).sum())
    return {"status":status,"capacity":capacity,"optimal_value":optimal_value,
            "baseline_value":baseline_value,
            "uplift_pct":100*(optimal_value/baseline_value-1),
            "allocation":allocation.to_dict()}

# Se resuelven los tres escenarios de capacidad: 60 %, 70 % y 85 % de la demanda
scenarios = [solve_scenario(parametros, capacity_ratio=r) for r in [0.60,0.70,0.85]]
# Se arma la tabla de resultados por escenario
scenario_df = pd.DataFrame([
    {"capacity_ratio": ratio, **{k:v for k,v in result.items() if k != "allocation"}}
    for ratio, result in zip([0.60,0.70,0.85], scenarios)
])
# solve_scenario devuelve uplift_pct como 40.07; Plotly necesita 0.4007 para el formato %.
scenario_df["improvement_pct"] = scenario_df["uplift_pct"] / 100
display(scenario_df)

,capacity_ratio,status,capacity,optimal_value,baseline_value,uplift_pct,improvement_pct
0,0.60,Optimal,5989,10629.73,7588.60,40.074981,0.400750
1,0.70,Optimal,6987,11478.03,8853.90,29.638126,0.296381
2,0.85,Optimal,8484,12214.08,10751.27,13.605927,0.136059


In [17]:
# Gráfico de la ventaja de optimizar según la capacidad disponible (60 %, 70 % y 85 %)
fig = px.bar(
    scenario_df,
    x="capacity_ratio",
    y="improvement_pct",
    text=scenario_df["improvement_pct"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "NA"),
    title="Ventaja de optimizar según capacidad disponible",
    labels={"capacity_ratio":"Capacidad / demanda", "improvement_pct":"Mejora vs intuitivo"}
)
fig.show()

### Pregunta 9
¿Por qué la mejora porcentual de PuLP cambia cuando la capacidad pasa de 60 % a 85 %?

**Respuesta:**

La mejora de PuLP frente a la regla intuitiva baja de **40.07 %** (capacidad 60 %) a **29.64 %** (70 %) y a **13.61 %** (85 %), aunque el valor óptimo absoluto sube (£10,629.73 → £11,478.03 → £12,214.08).

- **Capacidad escasa → decidir bien importa más.** Con 60 % hay que racionar mucho: la regla proporcional gasta unidades escasas en SKU de £0.29 y deja sin cubrir productos de £2.08 a £2.95, mientras que PuLP prioriza el valor por unidad.
- **Capacidad amplia → las dos reglas se parecen.** Con 85 % casi todo se puede servir, queda poco que racionar y ambas asignaciones se acercan. Si la capacidad igualara la demanda (100 %), los dos métodos asignarían lo mismo y la mejora sería 0.
- **Rendimientos decrecientes.** Los SKU de mayor precio ya están en su demanda, así que las unidades adicionales van a SKU más baratos: el valor marginal baja de £0.85 por unidad (60→70 %) a £0.49 (70→85 %).

La ventaja de optimizar es mayor cuanto más restringido está el sistema.

# 11. Decisión y recomendación

### Pregunta 10
¿Recomendaría operar con la asignación PuLP? Indique una condición que podría cambiar la respuesta.

**Respuesta:**

**Sí, recomiendo operar con la asignación de PuLP** como base de decisión. Con las mismas restricciones que la regla proporcional (capacidad, demanda, servicio mínimo y concentración), genera £11,478.03 frente a £8,853.90, es decir **+£2,624.13 (+29.64 %)**. Además, cumple todas las holguras y protege un servicio mínimo del 20 % en cada SKU.

**Condiciones que podrían cambiar la respuesta:**

- **Margen distinto del precio:** si los SKU de mayor precio (22086, 85099B, 23084) tuvieran un margen bajo o un costo de mantener inventario alto, el ranking de valor cambiaría y habría que optimizar utilidad, no ventas potenciales.
- **Demanda real menor a la estimada:** si la demanda de esos SKU en la semana de decisión fuera menor a la de la temporada alta usada (septiembre a diciembre), asignarles todo su "demanda" dejaría inventario sin vender.
- **Faltantes costosos:** si quedarse sin 84077, 22577 o 22578 (reducidos hasta el 20 %) provocara pérdida de clientes, habría que subir el servicio mínimo (ver Reto 2).

Antes de implementarlo conviene revisar esos tres puntos con datos de margen y de faltantes.

# 12. Retos individuales de aplicación

La parte guiada ya formuló y verificó el modelo. Resuelva los tres retos en su notebook personal. Son **10 preguntas** en total (4 + 3 + 3). En cada respuesta distinga resultados calculados, supuestos y recomendación. No altere las transacciones para producir una mejora mayor.

## Reto 1 — ¿Conviene ampliar la capacidad? (4 preguntas)

**Caso.** Operaciones puede aumentar la capacidad semanal, pero debe pagar por ella. Compare 60 %, 70 % y 85 % de la demanda total. Mantenga las demás reglas.

**R1.1.** Construya una tabla con capacidad, valor intuitivo, valor PuLP y mejora porcentual para los tres escenarios. ¿Qué representa cada cifra?

**R1.2.** Calcule el valor marginal por unidad adicional de capacidad entre 60 %→70 % y 70 %→85 %. ¿Por qué no son iguales?

**R1.3.** Si una unidad adicional de capacidad cuesta £0,65 por semana, ¿cuál ampliación conviene respecto del escenario de 60 %? Muestre valor incremental, costo incremental y saldo neto.

**R1.4.** Recomiende a operaciones una capacidad e indique una limitación de su cálculo.

In [18]:
# Reto 1: ¿Conviene ampliar la capacidad?
COSTO_UNIDAD = 0.65  # £ por unidad adicional de capacidad por semana (dato del caso, no proviene de UCI)

# R1.1 Tabla de los tres escenarios (scenario_df ya contiene los resultados de la sección 10)
r1 = scenario_df[["capacity_ratio", "capacity", "baseline_value", "optimal_value", "uplift_pct"]].copy()
r1.columns = ["capacidad_%_demanda", "capacidad_u", "valor_intuitivo_£", "valor_PuLP_£", "mejora_%"]
r1["capacidad_%_demanda"] = (r1["capacidad_%_demanda"] * 100).round().astype(int).astype(str) + " %"
display(r1.round(2))

# R1.2 y R1.3 Valor marginal, costo incremental y saldo neto de cada ampliación
base = scenario_df.set_index("capacity_ratio")
pasos = [(0.60, 0.70), (0.70, 0.85), (0.60, 0.85)]
filas = []
for a, b in pasos:
    d_cap = int(base.loc[b, "capacity"] - base.loc[a, "capacity"])
    d_val = float(base.loc[b, "optimal_value"] - base.loc[a, "optimal_value"])
    costo = COSTO_UNIDAD * d_cap
    filas.append({
        "ampliacion": f"{int(a*100)} % → {int(b*100)} %",
        "unidades_adicionales": d_cap,
        "valor_incremental_£": d_val,
        "valor_marginal_£/u": d_val / d_cap,
        "costo_incremental_£": costo,
        "saldo_neto_£": d_val - costo,
        "conviene": d_val - costo > 0,
    })
incremental = pd.DataFrame(filas)
display(incremental.round(4))

# Saldo neto total de cada escenario (valor PuLP − costo de toda la capacidad)
base["saldo_neto_£"] = base["optimal_value"] - COSTO_UNIDAD * base["capacity"]
display(base[["capacity", "optimal_value", "saldo_neto_£"]].round(2))

,capacidad_%_demanda,capacidad_u,valor_intuitivo_£,valor_PuLP_£,mejora_%
0,60 %,5989,7588.60,10629.73,40.07
1,70 %,6987,8853.90,11478.03,29.64
2,85 %,8484,10751.27,12214.08,13.61


,ampliacion,unidades_adicionales,valor_incremental_£,valor_marginal_£/u,costo_incremental_£,saldo_neto_£,conviene
0,60 % → 70 %,998,848.30,0.8500,648.70,199.6,True
1,70 % → 85 %,1497,736.05,0.4917,973.05,-237.0,False
2,60 % → 85 %,2495,1584.35,0.6350,1621.75,-37.4,False


,capacity,optimal_value,saldo_neto_£
capacity_ratio,,,
0.60,5989,10629.73,6736.88
0.70,6987,11478.03,6936.48
0.85,8484,12214.08,6699.48


In [19]:
# R1.4 Barrido fino de capacidad: ¿dónde el valor marginal deja de superar el costo de £0.65?
barrido = []
for r in np.arange(0.55, 0.9001, 0.01):
    s = solve_scenario(parametros, capacity_ratio=round(float(r), 2))
    barrido.append({
        "capacity_ratio": round(float(r), 2),
        "capacity": s["capacity"],
        "optimal_value": s["optimal_value"],
        "saldo_neto": s["optimal_value"] - COSTO_UNIDAD * s["capacity"],
    })
barrido = pd.DataFrame(barrido)
display(barrido.sort_values("saldo_neto", ascending=False).head(5).round(2))

mejor_ratio = barrido.loc[barrido["saldo_neto"].idxmax(), "capacity_ratio"]
print("Mejor capacidad entre las evaluadas en el barrido (pasos de 1 %):", mejor_ratio, "de la demanda")

# Verificación a nivel de unidad cerca del punto de quiebre: 22197 alcanza su demanda de 1,809 u. con 7,242 u. de capacidad
total_dem = int(parametros["demanda"].sum())
for cap in [6987, 7187, 7242, 7286]:
    s = solve_scenario(parametros, capacity_ratio=(cap + 0.5) / total_dem)
    print(f"capacidad {cap}: valor PuLP £{s['optimal_value']:.2f} · saldo neto £{s['optimal_value'] - COSTO_UNIDAD * s['capacity']:.2f}")
fig = px.line(barrido, x="capacity", y="saldo_neto",
              title="Saldo neto (valor PuLP − costo de capacidad) según capacidad",
              labels={"capacity": "Capacidad (unidades)", "saldo_neto": "Saldo neto (£)"})
fig.show()

,capacity_ratio,capacity,optimal_value,saldo_neto
18,0.73,7286,11718.98,6983.08
17,0.72,7187,11648.03,6976.48
19,0.74,7386,11773.98,6973.08
20,0.75,7486,11828.98,6963.08
16,0.71,7087,11563.03,6956.48


Mejor capacidad entre las evaluadas en el barrido (pasos de 1 %): 0.73 de la demanda
capacidad 6987: valor PuLP £11478.03 · saldo neto £6936.48
capacidad 7187: valor PuLP £11648.03 · saldo neto £6976.48
capacidad 7242: valor PuLP £11694.78 · saldo neto £6987.48
capacidad 7286: valor PuLP £11718.98 · saldo neto £6983.08


### Respuestas del Reto 1

**R1.1 — Tabla de escenarios**

| Capacidad / demanda | Capacidad (u.) | Valor intuitivo (£) | Valor PuLP (£) | Mejora |
|---|---|---|---|---|
| 60 % | 5,989 | 7,588.60 | 10,629.73 | 40.07 % |
| 70 % | 6,987 | 8,853.90 | 11,478.03 | 29.64 % |
| 85 % | 8,484 | 10,751.27 | 12,214.08 | 13.61 % |

- **Capacidad:** unidades semanales disponibles = `floor(% × 9,982)`.
- **Valor intuitivo:** valor potencial de venta (precio × unidades) al repartir la capacidad en proporción a la demanda, respetando mínimos, demanda y concentración.
- **Valor PuLP:** máximo valor potencial con exactamente las mismas restricciones.
- **Mejora:** `valor PuLP / valor intuitivo − 1`; mide cuánto vale decidir con optimización. Todo es valor potencial semanal, no utilidad.

**R1.2 — Valor marginal por unidad adicional**

- 60 % → 70 %: (11,478.03 − 10,629.73) ÷ (6,987 − 5,989) = 848.30 ÷ 998 = **£0.8500 por unidad**.
- 70 % → 85 %: (12,214.08 − 11,478.03) ÷ (8,484 − 6,987) = 736.05 ÷ 1,497 = **£0.4917 por unidad**.

No son iguales por **rendimientos decrecientes**: cada unidad adicional se asigna al SKU de mayor precio que todavía tiene espacio. En el primer tramo todos los SKU más caros ya están en su demanda, y las 998 unidades extra van a 22197 a £0.85. En el segundo tramo solo caben 255 unidades más en 22197 (llega a su demanda de 1,809) a £0.85; luego 612 unidades van a 22952 (£0.55) y 630 a SKU de £0.29: (255×0.85 + 612×0.55 + 630×0.29) ÷ 1,497 = £0.4917.

**R1.3 — Con costo de £0.65 por unidad adicional (frente a 60 %)**

| Ampliación | Unidades extra | Valor incremental | Costo incremental | Saldo neto |
|---|---|---|---|---|
| 60 % → 70 % | 998 | £848.30 | £648.70 | **+£199.60** |
| 60 % → 85 % | 2,495 | £1,584.35 | £1,621.75 | **−£37.40** |
| (70 % → 85 %) | 1,497 | £736.05 | £973.05 | −£237.00 |

Conviene ampliar a **70 %**: el valor marginal (£0.85) supera el costo (£0.65). Ampliar a 85 % no conviene frente a 60 %, porque el segundo tramo rinde en promedio £0.49 por unidad, menos que el costo.

**R1.4 — Recomendación**

Entre los tres escenarios evaluados recomiendo **70 % de capacidad (6,987 unidades)**: es el de mayor saldo neto (£6,936.48 frente a £6,736.88 en 60 % y £6,699.48 en 85 %). El barrido fino muestra que el máximo está cerca de **7,242 unidades (≈ 72.6 %)**, justo cuando 22197 alcanza su demanda; hasta ahí cada unidad rinde £0.85 > £0.65, y después rinde £0.55 < £0.65. Esa capacidad añade solo ≈ £51 de saldo neto sobre 70 %.

**Limitación:** el cálculo supone un costo lineal de £0.65 por unidad (dato del caso), y mide valor potencial de venta, no utilidad: no descuenta costo de compra ni margen. Además la demanda proviene de ventas observadas en una ventana de temporada alta, así que el beneficio real de ampliar puede ser menor.

## Reto 2 — Servicio mínimo y costo de la equidad (3 preguntas)

**Caso.** Comercial propone elevar el servicio mínimo para evitar productos casi desabastecidos. Evalúe 20 %, 30 % y 50 % con capacidad 70 %.

**R2.1.** Resuelva los tres escenarios y muestre valor óptimo y unidades por SKU. ¿Cuál sigue siendo factible?

**R2.2.** Calcule cuánto valor potencial se sacrifica al pasar de 20 % a 30 % y de 20 % a 50 %. Identifique qué SKU cambian más.

**R2.3.** Recomiende una política de servicio. Explique qué indicador operativo faltaría para decidirla en una empresa real.

In [20]:
# Reto 2: servicio mínimo y costo de la equidad (capacidad 70 %, concentración 35 %)
niveles_servicio = [0.20, 0.30, 0.50]
res_serv = {n: solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=n) for n in niveles_servicio}

# R2.1 Resumen por escenario: factibilidad y valor
resumen_serv = pd.DataFrame([{
    "servicio_min": f"{int(n*100)} %",
    "suma_minimos_u": int(np.maximum(np.ceil(n * parametros["demanda"]).astype(int), 1).sum()),
    "capacidad_u": r["capacity"],
    "estado": r["status"],
    "valor_PuLP_£": r["optimal_value"],
    "valor_intuitivo_£": r["baseline_value"],
    "mejora_%": r["uplift_pct"],
} for n, r in res_serv.items()]).round(2)
display(resumen_serv)

# Unidades por SKU en cada escenario factible
unid_serv = pd.DataFrame({f"{int(n*100)} %": pd.Series(r["allocation"])
                          for n, r in res_serv.items() if r["allocation"] is not None})
unid_serv = unid_serv.reindex(parametros.index)
unid_serv.insert(0, "precio_£", parametros["price"])
display(unid_serv)
print("Escenarios factibles:", [f"{int(n*100)} %" for n, r in res_serv.items() if r["status"] == "Optimal"])

# R2.2 Valor sacrificado frente a 20 % y SKU que más cambian
v20 = res_serv[0.20]["optimal_value"]
sacrificio = pd.DataFrame([{
    "cambio": f"20 % → {int(n*100)} %",
    "valor_sacrificado_£": v20 - res_serv[n]["optimal_value"],
    "sacrificio_%": 100 * (v20 - res_serv[n]["optimal_value"]) / v20,
} for n in [0.30, 0.50]]).round(2)
display(sacrificio)

cambios = pd.DataFrame({
    "precio_£": parametros["price"],
    "20→30 %": unid_serv["30 %"] - unid_serv["20 %"],
    "20→50 %": unid_serv["50 %"] - unid_serv["20 %"],
})
display(cambios.reindex(cambios["20→50 %"].abs().sort_values(ascending=False).index))

,servicio_min,suma_minimos_u,capacidad_u,estado,valor_PuLP_£,valor_intuitivo_£,mejora_%
0,20 %,1999,6987,Optimal,11478.03,8853.90,29.64
1,30 %,2997,6987,Optimal,11306.53,8854.46,27.69
2,50 %,4994,6987,Optimal,10791.63,8854.46,21.88


,precio_£,20 %,30 %,50 %
StockCode,,,,
22197,0.85,1554,1212,905
84077,0.29,245,367,612
85099B,2.08,1175,1175,1175
22086,2.95,1080,1080,1080
23084,2.08,959,959,959
84879,1.69,860,860,817
22952,0.55,153,230,383
22577,0.29,150,224,373
22578,0.29,139,208,347


Escenarios factibles: ['20 %', '30 %', '50 %']


,cambio,valor_sacrificado_£,sacrificio_%
0,20 % → 30 %,171.5,1.49
1,20 % → 50 %,686.4,5.98


,precio_£,20→30 %,20→50 %
StockCode,,,
22197,0.85,-342,-649
84077,0.29,122,367
21915,1.25,0,-336
22952,0.55,77,230
22577,0.29,74,223
22578,0.29,69,208
84879,1.69,0,-43
23084,2.08,0,0
85099B,2.08,0,0


### Respuestas del Reto 2

**R2.1 — Resultados con capacidad 70 % (6,987 u.)**

| Servicio mínimo | Suma de mínimos (u.) | Estado | Valor PuLP (£) |
|---|---|---|---|
| 20 % | 1,999 | Optimal | 11,478.03 |
| 30 % | 2,997 | Optimal | 11,306.53 |
| 50 % | 4,994 | Optimal | 10,791.63 |

| SKU | Precio (£) | 20 % | 30 % | 50 % |
|---|---|---|---|---|
| 22197 | 0.85 | 1,554 | 1,212 | 905 |
| 84077 | 0.29 | 245 | 367 | 612 |
| 85099B | 2.08 | 1,175 | 1,175 | 1,175 |
| 22086 | 2.95 | 1,080 | 1,080 | 1,080 |
| 23084 | 2.08 | 959 | 959 | 959 |
| 84879 | 1.69 | 860 | 860 | 817 |
| 22952 | 0.55 | 153 | 230 | 383 |
| 22577 | 0.29 | 150 | 224 | 373 |
| 22578 | 0.29 | 139 | 208 | 347 |
| 21915 | 1.25 | 672 | 672 | 336 |

**Los tres escenarios siguen siendo factibles:** la suma de mínimos más alta (4,994 u. al 50 %) cabe en la capacidad de 6,987, y el mínimo individual más alto (905 u. en 22197) está muy por debajo del tope de concentración de 2,445. En todos los casos las unidades asignadas suman 6,987.

**R2.2 — Costo de la equidad**

- 20 % → 30 %: se sacrifican **£171.50** (11,478.03 − 11,306.53), un **1.49 %** del valor.
- 20 % → 50 %: se sacrifican **£686.40** (11,478.03 − 10,791.63), un **5.98 %** del valor.

Los SKU que más cambian son los que están en los extremos de precio. **22197** (£0.85) cede 342 unidades al 30 % y 649 al 50 %, porque es el SKU de menor precio con holgura. Las unidades se desplazan hacia los SKU de £0.29 a £0.55: 84077 (+122 y +367), 22952 (+77 y +230), 22577 (+74 y +223) y 22578 (+69 y +208). Al 50 % también ceden 21915 (−336) y 84879 (−43). Los tres SKU de mayor precio (85099B, 22086 y 23084) no cambian en ningún escenario.

**R2.3 — Recomendación**

Recomiendo un **servicio mínimo de 30 %**: eleva en cerca de 50 % las unidades garantizadas de 84077, 22952, 22577 y 22578 (por ejemplo, 84077 pasa de 245 a 367) y cuesta solo £171.50 (1.49 %). Subir a 50 % cuesta £686.40 (5.98 %) y empieza a quitar inventario de productos de precio medio (21915 y 84879).

**Indicador operativo que faltaría:** el **costo de un faltante** (venta perdida más pérdida de clientes) o el **nivel de servicio objetivo** (por ejemplo, fill rate o probabilidad de quiebre) por SKU, junto con el **margen unitario**. Con eso se compararía el costo de la equidad (£171.50 o £686.40) contra las ventas y clientes que se evitan perder, en lugar de fijar un porcentaje arbitrario.

## Reto 3 — ¿La concentración limita de verdad? (3 preguntas)

**Caso.** Riesgos propone limitar cada SKU al 20 %, 25 %, 35 % o 50 % de la capacidad. Mantenga el escenario de capacidad 70 % y servicio mínimo 20 %.

**R3.1.** Resuelva las cuatro políticas. Presente valor óptimo, asignación y factibilidad.

**R3.2.** Indique en qué políticas la restricción cambia la solución y calcule el valor perdido frente a 35 %. ¿Qué demuestra el escenario de 50 %?

**R3.3.** Recomiende un límite de concentración y explique por qué el porcentaje por sí solo no mide todo el riesgo de inventario.

In [21]:
# Reto 3: ¿la concentración limita de verdad? (capacidad 70 %, servicio mínimo 20 %)
niveles_conc = [0.20, 0.25, 0.35, 0.50]
res_conc = {n: solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=n)
            for n in niveles_conc}

# R3.1 Resumen por política: factibilidad, valor y si el tope es vinculante
v35 = res_conc[0.35]["optimal_value"]
filas = []
for n, r in res_conc.items():
    limite = int(np.floor(n * r["capacity"]))
    a = pd.Series(r["allocation"]) if r["allocation"] is not None else None
    filas.append({
        "tope": f"{int(n*100)} %",
        "limite_u": limite,
        "estado": r["status"],
        "valor_PuLP_£": r["optimal_value"],
        "perdida_vs_35%_£": v35 - r["optimal_value"],
        "perdida_%": 100 * (v35 - r["optimal_value"]) / v35,
        "max_asignado_u": int(a.max()) if a is not None else None,
        "vinculante": bool((a >= limite).any()) if a is not None else None,
    })
resumen_conc = pd.DataFrame(filas).round(2)
display(resumen_conc)

# Asignación por SKU en cada política
unid_conc = pd.DataFrame({f"{int(n*100)} %": pd.Series(r["allocation"])
                          for n, r in res_conc.items() if r["allocation"] is not None})
unid_conc = unid_conc.reindex(parametros.index)
unid_conc.insert(0, "precio_£", parametros["price"])
display(unid_conc)

# R3.2 SKU que cambian frente a 35 %
dif_conc = unid_conc[["20 %", "25 %", "50 %"]].sub(unid_conc["35 %"], axis=0)
display(dif_conc[(dif_conc != 0).any(axis=1)])

# R3.3 Participación de cada SKU en unidades y en valor (solución con tope 35 %)
a35 = unid_conc["35 %"]
participacion = pd.DataFrame({
    "unidades": a35,
    "%_unidades": 100 * a35 / a35.sum(),
    "valor_£": a35 * parametros["price"],
    "%_valor": 100 * (a35 * parametros["price"]) / (a35 * parametros["price"]).sum(),
}).round(2)
display(participacion.sort_values("%_valor", ascending=False))

,tope,limite_u,estado,valor_PuLP_£,perdida_vs_35%_£,perdida_%,max_asignado_u,vinculante
0,20 %,1397,Optimal,11430.93,47.1,0.41,1397,True
1,25 %,1746,Optimal,11478.03,0.0,0.00,1554,False
2,35 %,2445,Optimal,11478.03,0.0,0.00,1554,False
3,50 %,3493,Optimal,11478.03,0.0,0.00,1554,False


,precio_£,20 %,25 %,35 %,50 %
StockCode,,,,,
22197,0.85,1397,1554,1554,1554
84077,0.29,245,245,245,245
85099B,2.08,1175,1175,1175,1175
22086,2.95,1080,1080,1080,1080
23084,2.08,959,959,959,959
84879,1.69,860,860,860,860
22952,0.55,310,153,153,153
22577,0.29,150,150,150,150
22578,0.29,139,139,139,139


,20 %,25 %,50 %
StockCode,,,
22197,-157,0,0
22952,157,0,0


,unidades,%_unidades,valor_£,%_valor
StockCode,,,,
22086,1080,15.46,3186.00,27.76
85099B,1175,16.82,2444.00,21.29
23084,959,13.73,1994.72,17.38
84879,860,12.31,1453.40,12.66
22197,1554,22.24,1320.90,11.51
21915,672,9.62,840.00,7.32
22952,153,2.19,84.15,0.73
84077,245,3.51,71.05,0.62
22577,150,2.15,43.50,0.38


### Respuestas del Reto 3

**R3.1 — Resultados con capacidad 70 % y servicio mínimo 20 %**

| Tope | Límite (u.) | Estado | Valor PuLP (£) | ¿Vinculante? |
|---|---|---|---|---|
| 20 % | 1,397 | Optimal | 11,430.93 | Sí (22197 = 1,397) |
| 25 % | 1,746 | Optimal | 11,478.03 | No |
| 35 % | 2,445 | Optimal | 11,478.03 | No |
| 50 % | 3,493 | Optimal | 11,478.03 | No |

Las cuatro políticas son factibles (el mayor mínimo es 362 u., por debajo de cualquier tope). La asignación es la misma del escenario base (22197 = 1,554) en 25 %, 35 % y 50 %. Con 20 % cambian solo dos SKU: 22197 baja a 1,397 y 22952 sube a 310.

**R3.2 — ¿Cuándo cambia la solución?**

La restricción **solo cambia la solución con el tope de 20 %**: 22197 cede 157 unidades (1,554 → 1,397) y 22952 las recibe (153 → 310). El valor perdido frente a 35 % es **£47.10** (157 × (0.85 − 0.55)), es decir **0.41 %**. Con 25 % y 50 % la pérdida es £0.

El escenario de 50 % **demuestra que relajar el tope no genera valor**: la solución es idéntica a la de 35 %. Lo que limita el valor son la capacidad y la demanda, no la concentración. La mayor participación que el modelo necesita es 22.2 % de la capacidad (1,554 ÷ 6,987); cualquier tope por encima de ese nivel es gratuito.

**R3.3 — Recomendación**

Recomiendo un **tope de 25 % (1,746 unidades por SKU)**: no cuesta nada frente a 35 % (£0) y reduce la exposición máxima por SKU de 2,445 a 1,746 unidades, con 192 unidades de holgura sobre la asignación actual. Si Riesgos exige más protección, el tope de 20 % cuesta £47.10 (0.41 %).

**Por qué el porcentaje por sí solo no mide todo el riesgo:**

- **Mide unidades, no valor.** Con tope 35 %, 22197 concentra 22.2 % de las unidades pero solo 11.5 % del valor, mientras que 22086 tiene 15.5 % de las unidades y **27.8 % del valor**. El riesgo económico está en los SKU caros.
- **No mide la incertidumbre de la demanda:** un SKU con ventas muy variables es más riesgoso que uno estable con la misma participación.
- **No ve la correlación:** productos sustitutos o de una misma familia pueden caer juntos, aunque cada uno esté bajo el tope.
- **No ve estacionalidad ni obsolescencia:** la ventana es de temporada alta; inventario de más puede quedar sin vender.
- **No ve el capital invertido:** el valor es precio de venta, no costo de compra.

# 13. Verificaciones finales

- Revise que cada solución factible respete capacidad, demanda, servicio y concentración.
- Indique si la regla intuitiva usa exactamente las mismas restricciones.
- Compruebe que las diez preguntas de los retos tengan respuesta numérica o una decisión justificada.
- Declare una limitación: ventas observadas ≠ demanda total; valor potencial ≠ utilidad.

In [22]:
# Verificaciones finales: cada solución factible debe respetar capacidad, demanda, servicio mínimo y concentración
def verificar(allocation, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=0.35):
    a = pd.Series(allocation).reindex(parametros.index)
    total = int(parametros["demanda"].sum())
    cap = int(np.floor(capacity_ratio * total))
    tope = int(np.floor(max_conc_ratio * cap))
    minimo = np.maximum(np.ceil(min_service_ratio * parametros["demanda"]).astype(int), 1)
    return {
        "capacidad": bool(a.sum() <= cap),
        "demanda": bool((a <= parametros["demanda"]).all()),
        "servicio_min": bool((a >= minimo).all()),
        "concentracion": bool((a <= tope).all()),
    }

filas = []
# Sensibilidad a capacidad (sección 10 / Reto 1)
for ratio, res in zip([0.60, 0.70, 0.85], scenarios):
    filas.append({"escenario": f"capacidad {int(ratio*100)} %", **verificar(res["allocation"], capacity_ratio=ratio)})
# Reto 2: servicio mínimo
for n, res in res_serv.items():
    filas.append({"escenario": f"servicio {int(n*100)} %", **verificar(res["allocation"], min_service_ratio=n)})
# Reto 3: concentración
for n, res in res_conc.items():
    filas.append({"escenario": f"concentración {int(n*100)} %", **verificar(res["allocation"], max_conc_ratio=n)})
# Línea base intuitiva del escenario principal: debe cumplir las mismas reglas que PuLP
filas.append({"escenario": "regla intuitiva (base)", **verificar(baseline["asignacion_intuitiva"])})

tabla_verif = pd.DataFrame(filas).set_index("escenario")
display(tabla_verif)
assert tabla_verif.all().all(), "Alguna solución viola una restricción."
print("Todas las soluciones factibles cumplen capacidad, demanda, servicio mínimo y concentración.")

,capacidad,demanda,servicio_min,concentracion
escenario,,,,
capacidad 60 %,True,True,True,True
capacidad 70 %,True,True,True,True
capacidad 85 %,True,True,True,True
servicio 20 %,True,True,True,True
servicio 30 %,True,True,True,True
servicio 50 %,True,True,True,True
concentración 20 %,True,True,True,True
concentración 25 %,True,True,True,True
concentración 35 %,True,True,True,True


Todas las soluciones factibles cumplen capacidad, demanda, servicio mínimo y concentración.


### Respuestas de las verificaciones finales

- **Restricciones:** la tabla de verificación comprueba, para las 10 soluciones de PuLP (3 de capacidad, 3 de servicio, 4 de concentración) y para la regla intuitiva, que ninguna excede la capacidad, la demanda ni el tope de concentración y que todas cumplen el servicio mínimo. Todas pasan.
- **¿La regla intuitiva usa las mismas restricciones?** Sí. `proportional_allocation` parte de los mínimos de servicio, reparte el resto en proporción a la demanda, limita cada SKU a su demanda y al tope de concentración, y asigna exactamente la capacidad (`assert alloc.sum() == capacity`). Dentro de `solve_scenario` se recalcula con los mismos parámetros de cada escenario, así que la comparación es justa. La única diferencia es el criterio de reparto: proporcional a la demanda frente a maximizar valor.
- **Diez preguntas de los retos:** R1.1 a R1.4, R2.1 a R2.3 y R3.1 a R3.3 tienen respuesta numérica (tablas, valores marginales, pérdidas en £ y %) y una decisión justificada.
- **Limitaciones declaradas:** (1) ventas observadas ≠ demanda total: hay demanda censurada por faltantes y la ventana es de temporada alta (septiembre a diciembre); (2) valor potencial ≠ utilidad: no hay costos, márgenes ni costo de faltante; (3) se usan 10 SKU y supuestos académicos (70 %, 20 %, 35 % y £0.65 por unidad), por lo que los resultados son un ejercicio de decisión y no una política lista para ejecutar.

# 14. Cierre ejecutivo y ticket de salida

Redacte máximo 12 líneas: decisión recomendada, capacidad, valor intuitivo y óptimo, mejora, costo de ampliar capacidad, política de servicio, concentración, principal riesgo y condición que cambiaría la recomendación.

**Ticket:** «Con ______ unidades de capacidad, PuLP genera £______ frente a £______ de la regla intuitiva. Recomiendo ______ porque ______. Esta conclusión cambiaría si ______».

---

## Cierre ejecutivo

1. **Decisión:** operar con la asignación de PuLP y no con la regla proporcional a la demanda.
2. **Capacidad:** 6,987 unidades (70 % de la demanda de 9,982 u.); no ampliar a 85 %.
3. **Valor:** PuLP genera £11,478.03 frente a £8,853.90 de la regla intuitiva: **+£2,624.13 (+29.64 %)**, con las mismas restricciones.
4. **Costo de ampliar (£0.65/u.):** de 60 % a 70 % el saldo neto es +£199.60; a 85 % es −£37.40. El máximo está cerca de 7,242 u. (≈ 72.6 %).
5. **Servicio:** recomiendo 30 % mínimo; cuesta £171.50 (1.49 %). Subir a 50 % cuesta £686.40 (5.98 %).
6. **Concentración:** el tope de 35 % no es vinculante (máximo real 22.2 %); recomiendo 25 % sin costo (20 % costaría £47.10).
7. **Política combinada (70 % / 30 % / 25 %):** £11,306.53 frente a £8,854.46 de la regla intuitiva, **+27.69 %**.
8. **Principal riesgo:** ventas observadas ≠ demanda y valor potencial ≠ utilidad; la ventana de 12 semanas es temporada alta.
9. **Condición que cambiaría la recomendación:** que el margen por SKU difiera del precio, que la demanda real de los SKU caros sea menor a la estimada, o que el costo de ampliar capacidad supere £0.85 por unidad.

**Ticket de salida:** «Con **6,987** unidades de capacidad, PuLP genera **£11,478.03** frente a **£8,853.90** de la regla intuitiva. Recomiendo **operar con la asignación PuLP al 70 % de capacidad (con servicio mínimo de 30 % y tope de concentración de 25 %)** porque **mejora el valor potencial en £2,624.13 (29.64 %) cumpliendo todas las restricciones, y las ampliaciones de capacidad a £0.65 por unidad solo se pagan hasta cerca del 72 % de la demanda**. Esta conclusión cambiaría si **el margen real de los SKU de mayor precio fuera bajo, si su demanda efectiva fuera menor a la estimada en temporada alta, o si el costo de ampliar capacidad superara £0.85 por unidad**».

# 15. Entrega y referencias

Suba el notebook ejecutado a `semana11/` de su repositorio personal y comparta el enlace de Colab. Incluya los tres retos y las diez respuestas individuales. Declare el uso de IA

Chen, D. (2012). *Online Retail II* [dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
Sharda, R., Delen, D. y Turban, E. *Business Intelligence, Analytics, Data Science and AI*.
Winston, W. L. *Operations Research: Applications and Algorithms*.